# Part 4 — Core API Deep-Dive (Human Example) 💻

*Last updated:* 2026-01-08

This notebook is a **hands-on tutorial** of the public IDTrack API using **human**.

**Learning objectives**
- Create the `idtrack.API` façade and understand what it wraps.
- Convert single identifiers (time travel + optional external outputs).
- Convert batches and summarize outcomes (1→0 / 1→1 / 1→n).
- Request explanation payloads for audit trails.
- Learn advanced knobs (external bridging, ambiguity strategy, assembly awareness).
- Learn introspection helpers (available databases, assemblies, releases, active ranges).

> **Prerequisite:** `initialization_graph.ipynb` (Part 3) is recommended so the graph loads from cache.


## 4.1 — The API Facade 💻

`idtrack.API` is the user-facing entry point. It handles:
- organism resolution (human/mouse/pig names and synonyms)
- building or loading a graph snapshot (the reproducible snapshot boundary)
- conversion helpers like `convert_identifier(...)` and `convert_identifier_multiple(...)`

In this notebook we build (or load) the **human** snapshot, then use it for the rest of the examples.

> **Expected result:** after the setup cell runs, `api.track` exists and conversions become available.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

organism, latest_release = api.resolve_organism('human')
SNAPSHOT_RELEASE = latest_release

api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=True)
print('Ready:', organism, 'snapshot', SNAPSHOT_RELEASE)


## 4.2 — Single Identifier Conversion 💻


In [ ]:
api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE)


If you see `no_corresponding=True`, it means the input could not be matched.
Try a different spelling/casing, or use an Ensembl ID directly.


### 4.2.1 Example: time travel (convert to an older release)

Why this matters: published datasets often use older releases.


In [ ]:
# Choose an older release to demonstrate time travel
older_release = SNAPSHOT_RELEASE - 10
api.convert_identifier('TP53', to_release=older_release)


### 4.2.2 Convert into an external database (HGNC)

To convert into a specific external database, pass `final_database=...`.
Database names are the same names you see in your external YAML.


In [ ]:
api.convert_identifier('ENSG00000141510', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')


### 4.2.3 How do I know which external databases are available?

Use the graph itself to list databases currently represented.


In [ ]:
g = api.track.graph
sorted(g.available_external_databases)[:50]


### 4.2.4 Understanding the result dictionary

Key fields:
- `query_id`: exactly what you typed
- `graph_id`: what IDTrack matched internally (normalization step)
- `target_id`: list of outputs (can be 0, 1, or many)
- `no_corresponding`: input didn’t match any node
- `no_conversion`: input matched, but no path to target release / database
- `no_target`: reached an Ensembl target, but requested external DB had no synonym

Important: `target_id` is a list because ambiguity is real and common.


### 4.2.5 Ambiguity control: strategy='best' vs strategy='all'

- `strategy='best'` (default): returns a single best target when possible
- `strategy='all'`: returns *all* candidates IDTrack found

Use `'all'` when you are doing QC or want to inspect ambiguous mappings.


In [ ]:
api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE, strategy='all')


## 4.3 — Batch Conversion 💻

Most workflows start from a list of identifiers (genes in a count matrix, markers, hits, etc.).
IDTrack provides two helpers:
- `convert_identifier_multiple(...)`
- `classify_multiple_conversion(...)` to summarize outcomes


In [ ]:
genes = ['TP53', 'BRCA1', 'BRCA2', 'BRAF', 'KRAS', 'NOT_A_REAL_GENE']
results = api.convert_identifier_multiple(genes, to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')
results[:2]  # show first two


In [ ]:
summary = api.classify_multiple_conversion(results)
# Each bin is a list of per-gene dictionaries
{k: len(v) for k, v in summary.items()}


If you want a human-readable report, you can print the summary bins:


In [ ]:
api.print_binned_conversion(summary)


## 4.4 — Explainability & Auditability 💻

When you set `explain=True`, the result includes a `the_path` field describing the graph edges followed.
This is very useful for advanced QC and debugging.


In [ ]:
explained = api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol', explain=True)
list(explained.keys())


In [ ]:
# The path dictionary keys are (target_id, ensembl_gene_id) pairs
list(explained['the_path'].keys())[:3]


`the_path` is intentionally detailed. For most users, the summary flags and `target_id` are enough.


## 4.5 — Advanced Conversion Options 💻

`api.convert_identifier(...)` is a convenience wrapper around `api.track.convert(...)`.

Use the high-level API most of the time.
But if you need full control (search settings, whether external bridging is allowed, deeper path diagnostics), you can call `Track.convert` directly.

### 4.5.1 Best vs all (selection strategy)

- `strategy='best'` returns a single globally best target.
- `strategy='all'` returns all scored targets (useful for ambiguity-aware pipelines).

### 4.5.2 Controlling external bridging

External bridging helps reconnect broken Ensembl histories using external IDs, but it can also increase search space.
Power users can toggle it via `go_external` on `Track.convert`.

### 4.5.3 Hyperconnected nodes

Some external identifiers connect to *many* entities (e.g. generic accessions). IDTrack detects these and limits their use to keep searches fast.

### 4.5.4 Assembly-aware conversions

If you need GRCh37-era mapping, build the human graph with `genome_assembly=37` (see Part 3). You can keep separate snapshots per assembly.

Example (advanced):
```python
api.track.convert(
    from_id='TP53',
    from_release=None,
    to_release=SNAPSHOT_RELEASE,
    final_database='HGNC Symbol',
    go_external=True,
    return_path=True,
)
```


In [ ]:
# Advanced demo: inspect hyperconnected nodes and compare `go_external` behavior.
# Safe: does not modify your cache; it only runs conversions.

# 1) Hyperconnected nodes (performance/ambiguity concept)
g = api.track.graph
hc = getattr(g, 'hyperconnective_nodes', {})
print('Hyperconnected external nodes:', len(hc))
if hc:
    top = sorted(hc.items(), key=lambda kv: kv[1], reverse=True)[:10]
    print('Top 10 by out-degree:')
    for node, deg in top:
        print(' ', deg, '-', node)

# 2) External bridging toggle (often matters when backbone history is disconnected)
# For many well-behaved genes, both calls will succeed; the point is the *option* exists.
res_no_external = api.track.convert(
    from_id='TP53',
    from_release=None,
    to_release=SNAPSHOT_RELEASE,
    final_database=None,
    go_external=False,
    prioritize_to_one_filter=True,
    return_path=False,
)
res_with_external = api.track.convert(
    from_id='TP53',
    from_release=None,
    to_release=SNAPSHOT_RELEASE,
    final_database=None,
    go_external=True,
    prioritize_to_one_filter=True,
    return_path=False,
)

print('go_external=False ->', 'OK' if res_no_external else None)
print('go_external=True  ->', 'OK' if res_with_external else None)


## 4.6 — Introspection & Discovery 💻

These helpers answer practical questions like:
- *Which external databases are available in my current graph?*
- *Which genome assemblies are represented?*
- *What Ensembl release range does my snapshot cover?*
- *When was a given identifier active across releases?*

The next cell demonstrates the most useful introspection calls.


In [ ]:
# Introspection demo

print('Assemblies in this graph:', sorted(api.list_genome_assemblies()))

ext_dbs = sorted(api.list_external_databases())
print('External DBs enabled (count):', len(ext_dbs))
print('External DBs (first 25):', ext_dbs[:25])

forms = api.external_database_forms()
print()
print('External DB connection forms (sample):')
for name in ext_dbs[:10]:
    print(' ', name, '→', forms.get(name))

rels = api.list_ensembl_releases()
print()
print('Ensembl releases in snapshot window:', (min(rels), max(rels)) if rels else None)

# Active ranges: when was an ID "alive" across releases?
# (Useful for provenance documentation.)
g = api.track.graph
example_gene = 'ENSG00000141510'  # TP53
if example_gene in g.nodes:
    print()
    print('Active ranges (main assembly) for', example_gene, ':', g.get_active_ranges_of_id.get(example_gene))
    try:
        print('Active ranges (all assemblies) for', example_gene, ':', g.get_active_ranges_of_id_ensembl_all_inclusive(example_gene))
    except Exception as e:
        print('All-assemblies active range failed ->', repr(e))
else:
    print('Example gene not found in graph (unexpected).')


## 4.7 — Practical advice (the kind that saves you a week) 💡

1. Always record your **snapshot boundary** (release) in your analysis notes.
2. If you share results, share the **external YAML** too.
3. When mapping is ambiguous, do not hide it — decide how your pipeline should handle 1→n mappings.
4. For scRNA-seq harmonization, prefer stable namespaces (Ensembl IDs) before switching to symbols.

> **Tip:** If you need troubleshooting checklists and diagnostics helpers, see `08_advanced_topics.ipynb` (Part 7.3).
